# 10 · Neural Network Experiments

**Learning objectives**
1. Turn a question about training into a controlled experiment: hypothesis, independent variable, controls, metrics, seeds and a decision rule.
2. Use held-out data for choices, not training data.
3. Read the results honestly, including what the step budget can and cannot show.

The experiment is `experiments/learning_rate_study/run.py`. This notebook reads its saved results and explains the design.

## 1. The question

**Question:** How does the learning rate affect validation performance of a $2\to4\to1$ network on a non-linear task?

| Element | Choice |
|---|---|
| Hypothesis | Learning rates that are too small do not converge within the budget; very large ones may oscillate. An intermediate value gives the best validation loss. |
| Independent variable | learning rate $\1\{0.05, 0.2, 0.5, 2.0, 8.0\}$ |
| Controls | data (fixed seed), architecture, 1500 full-batch steps, initialization scheme |
| Repetitions | initialization seeds 0 to 4 |
| Metrics | validation loss, validation accuracy, training loss, divergence count |
| Decision rule | pick the lowest mean validation loss among settings with no diverged run |

**Confounder to avoid:** changing the learning rate also changes how many updates a run makes in effect. Holding the number of steps fixed, as here, makes the comparison about speed within a budget, which is stated in the results.

## 2. The data

A circle task: label 1 inside radius $\1{1.5}$, with 5% of labels flipped. Training and validation sets are generated from different seeds, so the validation set is never used for training.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

import importlib.util
spec = importlib.util.spec_from_file_location("lrs", ROOT / "experiments" / "learning_rate_study" / "run.py")
lrs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lrs)

X_tr, y_tr = lrs.make_data(lrs.DATA_SEED, 120)
X_va, y_va = lrs.make_data(lrs.DATA_SEED + 1, 120)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", s=18, label="train")
ax.scatter(X_va[:, 0], X_va[:, 1], c=y_va, cmap="coolwarm", marker="x", s=24, label="validation")
th = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.sqrt(1.5) * np.cos(th), np.sqrt(1.5) * np.sin(th), "k--", lw=1)
ax.set_title("Circle task (dashed: noiseless boundary)"); ax.legend(fontsize=8); ax.set_aspect("equal")
plt.show()
print("train positives:", int(y_tr.sum()), "of", len(y_tr), "  validation positives:", int(y_va.sum()), "of", len(y_va))

train positives: 38 of 120   validation positives: 40 of 120


## 3. Results (saved run)

The table is read from `results/learning_rate_study.json`, produced by the script. Rerunning the script regenerates it.

In [2]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

res = json.loads((ROOT / "results" / "learning_rate_study.json").read_text(encoding="utf-8"))
print(f"{'lr':>6} {'train loss':>11} {'val loss (mean)':>16} {'val loss sd':>12} {'val acc':>8} {'diverged':>9}")
for s in res["summary"]:
    print(f"{s['learning_rate']:6} {s['train_loss_mean']:11.4f} {s['val_loss_mean']:16.4f} "
          f"{s['val_loss_sd']:12.4f} {s['val_acc_mean']:8.3f} {s['diverged_runs']:>6}/5")
print("chosen by decision rule:", res["chosen_learning_rate"])

lrs_ = [s["learning_rate"] for s in res["summary"]]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].errorbar(lrs_, [s["val_loss_mean"] for s in res["summary"]],
                 yerr=[s["val_loss_sd"] for s in res["summary"]], marker="o", capsize=4)
axes[0].set_xscale("log"); axes[0].set_title("Validation loss (mean ± sd, 5 seeds)")
axes[0].set_xlabel("learning rate"); axes[0].grid(True, which="both")
axes[1].plot(lrs_, [s["train_loss_mean"] for s in res["summary"]], marker="o", label="train loss")
axes[1].plot(lrs_, [s["val_loss_mean"] for s in res["summary"]], marker="s", label="validation loss")
axes[1].set_xscale("log"); axes[1].set_title("Train vs validation"); axes[1].legend(); axes[1].grid(True, which="both")
plt.tight_layout(); plt.show()

    lr  train loss  val loss (mean)  val loss sd  val acc  diverged
  0.05      0.6235           0.6365       0.0017    0.667      0/5
   0.2      0.5761           0.5966       0.0390    0.680      0/5
   0.5      0.3527           0.4261       0.0649    0.807      0/5
   2.0      0.2202           0.3519       0.0061    0.895      0/5
   8.0      0.2124           0.3944       0.0696    0.902      0/5
chosen by decision rule: 2.0


## 4. Reading the results

- **Smallest rate (0.05):** training loss is still about 0.62 after 1500 steps. It has not converged in this budget, so its validation loss mainly reflects speed.
- **Largest rates (2.0 and 8.0):** lowest validation loss near 2.0. At 8.0 the validation accuracy is slightly higher, but its loss is higher. The decision rule uses loss, and this disagreement is worth noting.
- **Spread:** five seeds give a rough spread. Differences smaller than the spread should not be interpreted.
- **Scope:** one dataset, one architecture, one step count. A different budget could change which rate looks best.

## 5. Checklist for any experiment in this repository

- [ ] Hypothesis and decision rule written before running.
- [ ] One independent variable changed; controls listed.
- [ ] Validation data used for decisions; training loss reported only as a convergence check.
- [ ] Several seeds, with mean and spread.
- [ ] Divergence counted explicitly.
- [ ] Results generated by a script that is in the repository.

## 6. Exercises

1. Double the number of steps to 3000 and rerun the script. Does the smallest rate still look worst? Report the change.
2. Add a learning rate of $0.01$ to the list. Predict its validation loss before running.
3. Replace the validation split with the training split and compare the chosen rate. Explain why that choice would be wrong.